# StockScope — Exploratory Data Analysis
Analyzing Reliance, TCS, and Infosys historical stock data.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_multiple
from feature_engineering import build_features

sns.set_theme(style='darkgrid')
%matplotlib inline

In [ ]:
# Load 2 years of data for all 3 stocks
stocks = load_multiple(period='2y')
for name, df in stocks.items():
    print(f'{name}: {df.shape}  |  {df.index[0].date()} → {df.index[-1].date()}')

In [ ]:
# Closing price comparison (normalized to 100)
fig, ax = plt.subplots(figsize=(14, 5))
for name, df in stocks.items():
    normalized = df['Close'] / df['Close'].iloc[0] * 100
    ax.plot(normalized, label=name)
ax.set_title('Normalized Close Price (Base = 100)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature engineering on TCS
tcs = build_features(stocks['TCS'])
tcs[['Close', 'SMA_20', 'SMA_50', 'EMA_20']].plot(figsize=(14, 5), title='TCS — Close vs Moving Averages')
plt.tight_layout()
plt.show()

In [ ]:
# Daily returns distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, df) in zip(axes, stocks.items()):
    featured = build_features(df)
    featured['Daily_Return'].hist(bins=60, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{name} Daily Returns')
    ax.axvline(0, color='red', linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
closes = pd.DataFrame({name: df['Close'] for name, df in stocks.items()}).dropna()
corr = closes.pct_change().corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Return Correlation Between Stocks')
plt.tight_layout()
plt.show()

In [ ]:
# Volatility comparison
fig, ax = plt.subplots(figsize=(14, 4))
for name, df in stocks.items():
    featured = build_features(df)
    (featured['Volatility_30d'] * 100).plot(ax=ax, label=name)
ax.set_title('30-Day Rolling Volatility (%)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Bollinger Bands for Reliance
rel = build_features(stocks['Reliance'])
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(rel['Close'], label='Close', color='black')
ax.plot(rel['BB_upper'], label='BB Upper', linestyle='--', color='red', alpha=0.6)
ax.plot(rel['BB_lower'], label='BB Lower', linestyle='--', color='green', alpha=0.6)
ax.fill_between(rel.index, rel['BB_lower'], rel['BB_upper'], alpha=0.1, color='gray')
ax.set_title('Reliance — Bollinger Bands')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Summary stats
summary = {}
for name, df in stocks.items():
    featured = build_features(df)
    summary[name] = {
        'Avg Daily Return (%)': round(featured['Daily_Return'].mean() * 100, 4),
        'Std Daily Return (%)': round(featured['Daily_Return'].std() * 100, 4),
        'Avg 30d Volatility (%)': round(featured['Volatility_30d'].mean() * 100, 4),
        'Total Return (%)': round((df['Close'].iloc[-1] / df['Close'].iloc[0] - 1) * 100, 2),
    }

pd.DataFrame(summary).T